## 3. Choose Final Model and Hyperparameter Tuning

In [2]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import average_precision_score, roc_auc_score

from src.utils import load_historical, make_pipeline, compare_topx_levels

In [3]:
X, y, df = load_historical("../data/historical_data.csv")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(9600, 16) (2400, 16)
0.10197916666666666 0.10208333333333333


In [4]:
final_model = LogisticRegression(max_iter=2000, solver="liblinear")

pipe = make_pipeline(final_model, X_train)

param_grid = {
    "model__C": [0.1, 1, 10],
    "model__class_weight": [None, "balanced"]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="average_precision",
    cv=5,
    n_jobs=-1
)

In [5]:
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV AP:", grid.best_score_)

Best params: {'model__C': 10, 'model__class_weight': None}
Best CV AP: 0.2842212848223595


In [6]:
best_pipe = grid.best_estimator_

y_proba_test = best_pipe.predict_proba(X_test)[:, 1]

print("Test AP:", average_precision_score(y_test, y_proba_test))
print("Test ROC-AUC:", roc_auc_score(y_test, y_proba_test))

Test AP: 0.27602488309617335
Test ROC-AUC: 0.7399327619678961


In [7]:
topx_results = compare_topx_levels(y_test, y_proba_test, top_fracs=(0.03, 0.05, 0.10))

topx_results

,top_frac,k_flagged,precision,recall
0,0.03,72,0.430556,0.126531
1,0.05,120,0.375000,0.183673
2,0.10,240,0.304167,0.297959


## Final Prioritization rule

We compared several prioritization levels (Top-3%, Top-5% and Top-10%) using the final tuned model.

The results show a clear trade-off between precision and recall:

- **Top-3%** gives the highest precision (0.43) but captures relatively few suspicious cases.
- **Top-10%** increase recall but significantly increases the manual review workload and lowers precision.
- **Top-5%** provides a good balance between review volume and precision.

Therefore, we chose **Top-5%** as the final prioritization rule.
This means that all events are ranked by predicted risk score and only the top 5% highest risk cases are sent to manual review by the Operations team.

## Risks and limitations

A lower Top-X level improves precision but also increase the number of suspicious cases that are not flagged (false negatives).  
Also, user behavior and fraud patterns may change over time. This means the models ranking performance should be monitord regularly and the chosen Top-X level may need adjustment if behavior changes.

### Hyperparameter tuning

We selected Logistic Regression as the final model because it achieved the best Average Precision during model comparison.

To improve the ranking quality, we tuned two hyperparameters:

- **C**: controls the strength of regularization. A larger value allows the model to fit the data more flexibly.
- **class_weight**: allows handling class imbalance.

We optimized these parameters using GridSearchCV with Average Precision as the evaluation metric. This aligns with the Operations requirement of prioritizing high-risk cases while keeping the review queue manageable.


### Final model

Logistic Regression was selected as the final model because it achieved the highest Average Precision during model comparison.

We performed hyperparameter tuning using GridSearchCV and optimized the parameters **C** and **class_weight** to improve ranking performance.

The final model achieved a **Test Average Precision of ~0.276** and **ROC-AUC of ~0.74**, which provides a solid ranking of high-risk cases for prioritization.


In [8]:
from src.utils import save_pipeline
import os
os.makedirs("../models", exist_ok=True)

save_pipeline(best_pipe, "../models/final_model.joblib")

print("Pipeline saved to ../models/final_model.joblib")

Pipeline saved to ../models/final_model.joblib
